<a href="https://colab.research.google.com/github/ihaseeb0081/flyrank-ml-internship-haseeb/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihaseeb0081/flyrank-ml-internship-haseeb/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method Choice and Why

### Method Choice and Why

For this capstone, I selected a Random Forest Regressor.

My lane is Content Refresh Opportunity Scoring. The goal is to assign a continuous opportunity score to webpages so that pages can be prioritized for content review.

Random Forest is suitable because it can capture non-linear relationships between search performance, content freshness, and engagement signals. It is also useful for interpreting which features contribute most to the model output.

The model is used as decision-support for prioritizing content refresh opportunities rather than as an automated decision maker.

In [48]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

print("Random Forest model initialized successfully.")

Random Forest model initialized successfully.


In [49]:
!git clone https://github.com/ihaseeb0081/flyrank-ml-internship-haseeb.git

fatal: destination path 'flyrank-ml-internship-haseeb' already exists and is not an empty directory.


In [50]:
!ls -lah /content/flyrank-ml-internship-haseeb/data/raw/

total 6.5M
drwxr-xr-x 2 root root 4.0K Sep  6 15:44 .
drwxr-xr-x 3 root root 4.0K Sep  6 15:44 ..
-rw-r--r-- 1 root root 6.5M Sep  6 15:44 content_refresh_anonymized.csv


In [51]:
import pandas as pd
import numpy as np

data_path = "/content/flyrank-ml-internship-haseeb/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (30000, 44)


In [52]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [53]:
# Create the components used in the ML-07 baseline score

staleness_score = (
    df["days_since_last_update"] /
    df["days_since_last_update"].max()
) * 100

search_score = (
    df["search_volume"].fillna(0) /
    df["search_volume"].max()
) * 100

trend_score = np.where(
    df["trend_direction"].eq("down"),
    100,
    0
)

# ML-07 baseline score
df["baseline_score"] = (
    0.40 * staleness_score
    + 0.30 * search_score
    + 0.30 * trend_score
)

print("Baseline score created successfully.")
print(df["baseline_score"].describe())

Baseline score created successfully.
count    30000.000000
mean        21.264630
std         15.964208
min          0.107239
25%          2.363303
50%         31.052503
75%         32.363303
max         70.000000
Name: baseline_score, dtype: float64


In [54]:
# Select features for the Random Forest model

feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

X = df[feature_columns].copy()

# Fill missing numerical values with the median
X = X.fillna(X.median(numeric_only=True))

# Target variable
y = df["baseline_score"]

print("Features and target created successfully.")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of features:", len(feature_columns))

Features and target created successfully.
X shape: (30000, 11)
y shape: (30000,)
Number of features: 11


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split Design

I used a client-grouped 80/20 train-test split.

Pages from the same client are kept on the same side of the split. This reduces the risk of client-specific patterns appearing in both training and testing data.

This provides a more honest estimate of how the model generalizes to unseen client groups.

The same test set is used when evaluating the model and comparing it with the baseline.

In [55]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training clients:", df.iloc[train_idx]["client_id"].nunique())
print("Testing clients:", df.iloc[test_idx]["client_id"].nunique())

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + Compare vs My Baseline

I trained a Random Forest Regressor on the training portion of the dataset.

The model was evaluated on the held-out test set using MAE, RMSE, and R².

The baseline and model are evaluated on the same test pages so that the comparison is consistent.

The results are interpreted as decision-support evidence rather than as a claim of production-level performance.

In [56]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train the Random Forest model
model.fit(X_train, y_train)

# Predict on unseen test data
y_pred = model.predict(X_test)

# Model metrics
model_mae = mean_absolute_error(y_test, y_pred)
model_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
model_r2 = r2_score(y_test, y_pred)

print("Random Forest Results")
print("---------------------")
print(f"MAE : {model_mae:.4f}")
print(f"RMSE: {model_rmse:.4f}")
print(f"R²  : {model_r2:.4f}")

Random Forest Results
---------------------
MAE : 14.0202
RMSE: 15.4854
R²  : -0.0308


In [57]:
# Baseline predictions for the same test pages
baseline_test = y_test.values

baseline_mae = mean_absolute_error(y_test, baseline_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_test))
baseline_r2 = r2_score(y_test, baseline_test)

print("Baseline Results")
print("----------------")
print(f"MAE : {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²  : {baseline_r2:.4f}")

Baseline Results
----------------
MAE : 0.0000
RMSE: 0.0000
R²  : 1.0000


In [58]:
results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Baseline": [
        baseline_mae,
        baseline_rmse,
        baseline_r2
    ],
    "Random Forest": [
        model_mae,
        model_rmse,
        model_r2
    ]
})

results

,Metric,Baseline,Random Forest
0,MAE,0.0,14.020169
1,RMSE,0.0,15.485352
2,R²,1.0,-0.030799


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and Interpretation

The model predictions were compared with the baseline refresh opportunity scores on the held-out test set.

The measured errors show how closely the Random Forest reproduces the baseline scoring signal.

Feature importance was examined to understand which measured variables the model relies on most.

Because the target variable is derived from the baseline scoring rules, the Random Forest results should not be interpreted as proof that the machine learning model is superior to the baseline.

The output is directional decision-support for prioritizing content refresh opportunities.

In [59]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("Top features used by the model:")
display(feature_importance)

Top features used by the model:


,Feature,Importance
5,impressions_90d,0.196335
8,days_since_last_update,0.186222
10,avg_position,0.152399
4,char_count,0.086801
7,sessions_90d,0.083440
3,word_count,0.079832
9,ctr,0.064939
6,clicks_90d,0.043407
0,search_volume,0.043257
1,competition,0.038121


In [60]:
error_analysis = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

error_analysis["Absolute_Error"] = (
    np.abs(error_analysis["Actual"] - error_analysis["Predicted"])
)

print("Error summary:")
display(error_analysis["Absolute_Error"].describe())

print("Largest prediction errors:")
display(
    error_analysis
    .sort_values("Absolute_Error", ascending=False)
    .head(10)
)

Error summary:


,Absolute_Error
count,6.163000e+03
mean,1.402017e+01
std,6.575563e+00
min,1.776357e-15
25%,9.230198e+00
50%,1.388806e+01
75%,1.868151e+01
max,3.000130e+01


Largest prediction errors:


,Actual,Predicted,Absolute_Error
2285,32.363303,2.362006,30.001297
4686,11.160923,41.160842,29.999919
2712,32.144772,2.244664,29.900109
2077,11.053685,40.860558,29.806874
466,11.152815,40.903298,29.750483
406,32.152880,2.453367,29.699514
503,11.152815,40.574950,29.422135
211,32.363303,2.959865,29.403438
3441,11.156869,40.557171,29.400302
5637,32.148826,2.748988,29.399838


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.